# Cluster Profiling

This notebook profiles the customer segments identified by K-Means clustering. It provides interpretable descriptions of each cluster through radar charts, box plots, top products analysis, and country composition to support actionable business insights.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from math import pi

## Helper Classes

In [ ]:
class ClusterProfiler:
    def __init__(self, str_uri_clustered, str_uri_transactions, str_dirname_output, list_features):
        self.str_uri_clustered = str_uri_clustered
        self.str_uri_transactions = str_uri_transactions
        self.str_dirname_output = str_dirname_output
        self.list_features = list_features

    def import_data(self):
        self.df = pd.read_parquet(self.str_uri_clustered)
        self.df_txn = pd.read_parquet(self.str_uri_transactions)
        self.df_txn['invoicedate'] = pd.to_datetime(self.df_txn['invoicedate'], errors='coerce')
        self.int_k = self.df['cluster'].nunique()
        print(f'Customers: {self.df.shape[0]:,}')
        print(f'Transactions: {self.df_txn.shape[0]:,}')
        print(f'Clusters: {self.int_k}')

    def summary_table(self):
        list_agg = {}
        for col in self.list_features:
            list_agg[col] = ['mean', 'median', 'std']
        list_agg['customer_id'] = 'count'
        df_summary = self.df.groupby(by='cluster').agg(list_agg)
        df_summary.columns = ['_'.join(col).strip() for col in df_summary.columns.values]
        df_summary.rename(columns={'customer_id_count': 'n_customers'}, inplace=True)
        df_summary['pct_customers'] = df_summary['n_customers'] / df_summary['n_customers'].sum()
        self.df_summary = df_summary
        df_summary.to_csv(f'{self.str_dirname_output}/cluster_summary.csv')
        print(f'Saved cluster summary to {self.str_dirname_output}/cluster_summary.csv')
        return df_summary

    def plot_radar_chart(self):
        # normalize features to 0-1 for radar
        df_means = self.df.groupby(by='cluster')[self.list_features].mean()
        df_norm = df_means.copy()
        for col in self.list_features:
            flt_min = df_means[col].min()
            flt_max = df_means[col].max()
            if flt_max > flt_min:
                df_norm[col] = (df_means[col] - flt_min) / (flt_max - flt_min)
            else:
                df_norm[col] = 0.5

        int_n_features = len(self.list_features)
        angles = [n / float(int_n_features) * 2 * pi for n in range(int_n_features)]
        angles += angles[:1]

        fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
        ax.set_title('Cluster Profiles (Radar Chart)', fontsize=14, y=1.08)

        colors = plt.cm.Set2(np.linspace(0, 1, self.int_k))
        for cluster_id in range(self.int_k):
            values = df_norm.loc[cluster_id].values.tolist()
            values += values[:1]
            ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {cluster_id}', color=colors[cluster_id])
            ax.fill(angles, values, alpha=0.1, color=colors[cluster_id])

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(self.list_features, fontsize=9)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/radar_chart.png', bbox_inches='tight', dpi=150)
        plt.show()

    def plot_box_plots(self):
        int_n_features = len(self.list_features)
        int_ncols = 4
        int_nrows = (int_n_features + int_ncols - 1) // int_ncols
        fig, axes = plt.subplots(int_nrows, int_ncols, figsize=(20, 5 * int_nrows))
        fig.suptitle('Feature Distributions by Cluster (Box Plots)', fontsize=16, y=1.02)
        axes_flat = axes.flatten()

        for idx, col in enumerate(self.list_features):
            ax = axes_flat[idx]
            p01 = self.df[col].quantile(0.01)
            p99 = self.df[col].quantile(0.99)
            df_clipped = self.df[[col, 'cluster']].copy()
            df_clipped[col] = df_clipped[col].clip(lower=p01, upper=p99)
            sns.boxplot(data=df_clipped, x='cluster', y=col, ax=ax, palette='Set2')
            ax.set_title(col)
            ax.set_xlabel('Cluster')

        # hide empty axes
        for idx in range(int_n_features, len(axes_flat)):
            axes_flat[idx].set_visible(False)

        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/box_plots.png', bbox_inches='tight', dpi=150)
        plt.show()

    def plot_country_composition(self):
        # merge cluster assignments with transactions
        df_txn = self.df_txn.copy()
        df_txn = df_txn[df_txn['customer_id'].notna()].copy()
        df_txn = df_txn.merge(self.df[['customer_id', 'cluster']], on='customer_id', how='inner')

        # top countries per cluster
        df_country = df_txn.groupby(by=['cluster', 'country'], as_index=False).agg(
            n_customers=('customer_id', 'nunique'),
        )

        fig, axes = plt.subplots(1, self.int_k, figsize=(6 * self.int_k, 6))
        fig.suptitle('Top 10 Countries by Cluster', fontsize=16, y=1.02)
        if self.int_k == 1:
            axes = [axes]

        for cluster_id in range(self.int_k):
            ax = axes[cluster_id]
            df_tmp = df_country[df_country['cluster'] == cluster_id].copy()
            df_tmp.sort_values(by='n_customers', ascending=True, inplace=True)
            df_tmp = df_tmp.tail(10)
            ax.barh(df_tmp['country'], df_tmp['n_customers'], color='tab:blue', alpha=0.8)
            ax.set_title(f'Cluster {cluster_id}')
            ax.set_xlabel('N Customers')

        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/country_composition.png', bbox_inches='tight', dpi=150)
        plt.show()

    def plot_top_products(self, int_top_n=10):
        # merge cluster assignments with transactions
        df_txn = self.df_txn.copy()
        df_txn = df_txn[df_txn['customer_id'].notna()].copy()
        df_txn = df_txn[~df_txn['invoice'].astype(str).str.startswith('C')].copy()
        df_txn = df_txn.merge(self.df[['customer_id', 'cluster']], on='customer_id', how='inner')
        df_txn['revenue'] = df_txn['quantity'] * df_txn['price']

        fig, axes = plt.subplots(1, self.int_k, figsize=(6 * self.int_k, 6))
        fig.suptitle(f'Top {int_top_n} Products by Revenue per Cluster', fontsize=16, y=1.02)
        if self.int_k == 1:
            axes = [axes]

        for cluster_id in range(self.int_k):
            ax = axes[cluster_id]
            df_tmp = df_txn[df_txn['cluster'] == cluster_id].copy()
            df_prod = df_tmp.groupby(by='description', as_index=False).agg(
                total_revenue=('revenue', 'sum'),
            )
            df_prod.sort_values(by='total_revenue', ascending=True, inplace=True)
            df_prod = df_prod.tail(int_top_n)
            ax.barh(df_prod['description'], df_prod['total_revenue'], color='tab:green', alpha=0.8)
            ax.set_title(f'Cluster {cluster_id}')
            ax.set_xlabel('Total Revenue')

        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/top_products.png', bbox_inches='tight', dpi=150)
        plt.show()

    def plot_purchase_timing(self):
        # merge cluster assignments with transactions
        df_txn = self.df_txn.copy()
        df_txn = df_txn[df_txn['customer_id'].notna()].copy()
        df_txn = df_txn[~df_txn['invoice'].astype(str).str.startswith('C')].copy()
        df_txn = df_txn.merge(self.df[['customer_id', 'cluster']], on='customer_id', how='inner')
        df_txn['day_of_week'] = df_txn['invoicedate'].dt.day_name()
        df_txn['hour'] = df_txn['invoicedate'].dt.hour

        list_day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle('Purchase Timing by Cluster', fontsize=16, y=1.02)

        # day of week
        ax = axes[0]
        for cluster_id in sorted(self.df['cluster'].unique()):
            df_tmp = df_txn[df_txn['cluster'] == cluster_id].copy()
            ser_counts = df_tmp['day_of_week'].value_counts()
            ser_counts = ser_counts.reindex(list_day_order, fill_value=0)
            ser_pct = ser_counts / ser_counts.sum()
            ax.plot(list_day_order, ser_pct.values, marker='o', markersize=4, label=f'Cluster {cluster_id}')
        ax.set_title('Day of Week')
        ax.set_xlabel('Day')
        ax.set_ylabel('Proportion of Transactions')
        ax.legend()
        ax.tick_params(axis='x', rotation=45)

        # hour of day
        ax = axes[1]
        for cluster_id in sorted(self.df['cluster'].unique()):
            df_tmp = df_txn[df_txn['cluster'] == cluster_id].copy()
            ser_counts = df_tmp['hour'].value_counts().sort_index()
            ser_pct = ser_counts / ser_counts.sum()
            ax.plot(ser_pct.index, ser_pct.values, marker='o', markersize=4, label=f'Cluster {cluster_id}')
        ax.set_title('Hour of Day')
        ax.set_xlabel('Hour')
        ax.set_ylabel('Proportion of Transactions')
        ax.legend()

        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/purchase_timing.png', bbox_inches='tight', dpi=150)
        plt.show()

    def plot_revenue_over_time(self):
        df_txn = self.df_txn.copy()
        df_txn = df_txn[df_txn['customer_id'].notna()].copy()
        df_txn = df_txn[~df_txn['invoice'].astype(str).str.startswith('C')].copy()
        df_txn = df_txn.merge(self.df[['customer_id', 'cluster']], on='customer_id', how='inner')
        df_txn['revenue'] = df_txn['quantity'] * df_txn['price']
        df_txn['invoice_month'] = df_txn['invoicedate'].dt.to_period('M').dt.to_timestamp()

        df_pivot = df_txn.groupby(by=['invoice_month', 'cluster'], as_index=False).agg(
            total_revenue=('revenue', 'sum'),
        )

        fig, ax = plt.subplots(figsize=(14, 5))
        ax.set_title('Monthly Revenue by Cluster')
        ax.set_xlabel('Month')
        ax.set_ylabel('Total Revenue')

        for cluster_id in sorted(self.df['cluster'].unique()):
            df_tmp = df_pivot[df_pivot['cluster'] == cluster_id].copy()
            ax.plot(df_tmp['invoice_month'], df_tmp['total_revenue'],
                    marker='o', markersize=3, label=f'Cluster {cluster_id}')

        ax.legend()
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/revenue_over_time.png', bbox_inches='tight', dpi=150)
        plt.show()

## Constants

In [ ]:
str_bucket = 'cluster-analysis-demo'
print(f'Bucket: {str_bucket}')

str_task = '04_profiling'
print(f'Task: {str_task}')

str_dirname_output = './output'

# data uris
str_uri_clustered = f's3://{str_bucket}/03_clustering/df_clustered.parquet'
str_uri_transactions = f's3://{str_bucket}/00_data_collection/data.parquet'

# features
list_features = [
    'recency', 'frequency', 'monetary', 'avg_order_value',
    'avg_items_per_order', 'unique_products', 'avg_unit_price', 'tenure_days',
]

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

## Initialize Profiler

In [ ]:
cls_profiler = ClusterProfiler(
    str_uri_clustered=str_uri_clustered,
    str_uri_transactions=str_uri_transactions,
    str_dirname_output=str_dirname_output,
    list_features=list_features,
)

## Import Data

In [ ]:
cls_profiler.import_data()

## Cluster Summary Table

In [ ]:
cls_profiler.summary_table()

## Radar Chart

In [ ]:
cls_profiler.plot_radar_chart()

## Box Plots

In [ ]:
cls_profiler.plot_box_plots()

## Country Composition

In [ ]:
cls_profiler.plot_country_composition()

## Top Products by Cluster

In [ ]:
cls_profiler.plot_top_products()

## Purchase Timing

In [ ]:
cls_profiler.plot_purchase_timing()

## Revenue Over Time by Cluster

In [ ]:
cls_profiler.plot_revenue_over_time()